In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

# data

In [ ]:
DAY = 0

In [ ]:
df = pd.read_csv(f"./round-3-island-data-bottle/prices_round_3_day_{DAY}.csv", sep=";", header=0)

In [ ]:
def swmid(row):
    bid = row['bid_price_1']
    ask = row['ask_price_1']
    bid_volume = row['bid_volume_1']
    ask_volume = row['ask_volume_1']
    return ((bid*ask_volume + ask*bid_volume)/(bid_volume + ask_volume))

def mm_mid_basket(row, volume_cutoff=10):
    # Find the best bid with volume >= volume_cutoff
    for i in range(1, 4):
        if row[f'bid_volume_{i}'] >= volume_cutoff:
            best_bid = row[f'bid_price_{i}']
            break
    else:
        # No bid with sufficient volume found
        best_bid = None

    # Find the best ask with volume >= volume_cutoff
    for i in range(1, 4):
        if row[f'ask_volume_{i}'] >= volume_cutoff:
            best_ask = row[f'ask_price_{i}']
            break
    else:
        # No ask with sufficient volume found
        best_ask = None

    # Calculate the mid-price if both best bid and best ask are found
    if best_bid is not None and best_ask is not None:
        mid_price = (best_bid + best_ask) / 2
        return mid_price
    else:
        # Return the mid_price column value as default
        return row['mid_price']
    

In [ ]:
def fair_price(row):
    if row['product'] == 'GIFT_BASKET':
        return mm_mid_basket(row, volume_cutoff=10)
    else:
        return swmid(row)

df['fair'] = df.apply(fair_price, axis=1)

In [ ]:
# Define the weights dictionary
weights = {
    'CHOCOLATE': 4,
    'STRAWBERRIES': 6,
    'ROSES': 1
}

# Get the unique product names
products = df['product'].unique()

# Create a new dataframe 'df_fairs' with the desired columns
columns = ['timestamp'] + list(products) + ['SYNTHETIC']
df_fairs = pd.DataFrame(columns=columns)

# Iterate over unique timestamps in the original dataframe
for timestamp in df['timestamp'].unique():
    # Get the rows for the current timestamp
    rows = df[df['timestamp'] == timestamp]
    
    # Create a dictionary to store the fair values for each product
    fairs = {}
    
    # Iterate over each product and extract its fair value
    for product in products:
        fair = rows.loc[rows['product'] == product, 'fair'].values[0]
        fairs[product] = fair
    
    # Calculate the synthetic fair
    synthetic_fair = sum(fairs[product] * weights.get(product, 0) for product in ['CHOCOLATE', 'STRAWBERRIES', 'ROSES'])
    
    # Create a new row as a DataFrame
    new_row = pd.DataFrame({'timestamp': [timestamp], **{product: [fairs[product]] for product in products}, 'SYNTHETIC': [synthetic_fair]})
    
    # Concatenate the new row with df_fairs
    df_fairs = pd.concat([df_fairs, new_row], ignore_index=True)

# Reset the index of df_fairs (optional)
df_fairs = df_fairs.reset_index(drop=True)

In [ ]:
df_fairs

## boellinger band

UFCK fuck fuck fuck fuci kuf ckf uckf cu fk 

In [ ]:
df_fairs = df_fairs[['timestamp', "CHOCOLATE", "STRAWBERRIES", "ROSES", "GIFT_BASKET", 'SYNTHETIC']]

In [ ]:
df_fairs['BASKET_MINUS_SYNTHETIC'] = df_fairs['GIFT_BASKET'] - df_fairs["SYNTHETIC"]

In [ ]:
import pandas as pd
import plotly.graph_objects as go

def bollinger_bands(data, column='BASKET_MINUS_SYNTHETIC', window=20, num_std=2):
    """
    Calculate Bollinger Bands for a given DataFrame and column.
    
    Parameters:
    data (pd.DataFrame): The input DataFrame.
    column (str): The name of the column to calculate Bollinger Bands for.
    window (int): The rolling window size for the Simple Moving Average (SMA).
    num_std (int): The number of standard deviations for the upper and lower bands.
    
    Returns:
    pd.DataFrame: The input DataFrame with additional columns for the Bollinger Bands.
    """
    # Calculate the Simple Moving Average (SMA)
    data['SMA'] = data[column].rolling(window=window).mean()
    
    # Calculate the standard deviation
    data['STD'] = data[column].rolling(window=window).std()
    
    # Calculate the upper and lower Bollinger Bands
    data['UPPER_BAND'] = data['SMA'] + (data['STD'] * num_std)
    data['LOWER_BAND'] = data['SMA'] - (data['STD'] * num_std)
    
    return data

# Example usage
df_fairs = bollinger_bands(df_fairs, window=200, num_std=1.5)

# Calculate the change in STD
df_fairs['STD_CHANGE'] = df_fairs['STD'].diff()

# Create the plot
fig = go.Figure()

# Add the BASKET_MINUS_SYNTHETIC line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['BASKET_MINUS_SYNTHETIC'].ewm(halflife=5).mean(),
                         mode='lines', name='BASKET_MINUS_SYNTHETIC'))

# Add the Upper Bollinger Band line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['UPPER_BAND'],
                         mode='lines', name='Upper Band', line=dict(color='red')))

# Add the Simple Moving Average (SMA) line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['SMA'],
                         mode='lines', name='SMA', line=dict(color='blue')))

# Add the Lower Bollinger Band line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['LOWER_BAND'],
                         mode='lines', name='Lower Band', line=dict(color='green')))

# Add the change in STD line on a separate y-axis
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['STD'],
                         mode='lines', name='STD', line=dict(color='purple'),
                         yaxis='y2'))

# Update the layout to include the secondary y-axis
fig.update_layout(
    title='Bollinger Bands with STD',
    xaxis_title='Timestamp',
    yaxis=dict(title='BASKET_MINUS_SYNTHETIC'),
    yaxis2=dict(title='STD', overlaying='y', side='right')
)

# Display the plot
fig.show()

In [ ]:
def identify_regime(data, mean=370, lookback=20, ewma=0):
    """
    Identify the regime based on the price's interaction with Bollinger Bands.

    Parameters:
    data (pd.DataFrame): The input DataFrame containing Bollinger Bands.
    lookback (int): The lookback period to determine the regime.

    Returns:
    pd.Series: A series containing the identified regime for each row.
    """
    price = data['BASKET_MINUS_SYNTHETIC']
    if ewma != 0: 
        price = data['BASKET_MINUS_SYNTHETIC'].ewm(halflife=ewma).mean()
    upper = data['UPPER_BAND']
    lower = data['LOWER_BAND']

    # Create a rolling window of size lookback
    rolling_window = price.rolling(window=lookback)

    # Check if price is above upper band or below lower band
    above_upper = (price >= upper).rolling(window=lookback).sum()
    below_lower = (price <= lower).rolling(window=lookback).sum()

    # Count contiguous touches of upper and lower bands
    touched_upper = (above_upper > 1).astype(int)
    touched_lower = (below_lower > 1).astype(int)

    # Assign regimes based on the touched bands
    regimes = pd.Series(index=data.index, dtype='object')
    regimes.loc[(touched_upper == 1) & (touched_lower == 1)] = 'OSCILLATING'
    regimes.loc[(touched_upper == 1) & (touched_lower == 0)] = 'RIDING_UPPER'
    regimes.loc[(touched_upper == 0) & (touched_lower == 1)] = 'RIDING_LOWER'
    regimes.fillna('NEUTRAL', inplace=True)

    # Fill the initial lookback period with 'NEUTRAL'
    regimes.iloc[:lookback] = 'NEUTRAL'

    return regimes

# Apply the identify_regime function to your DataFrame
df_fairs['REGIME'] = identify_regime(df_fairs, mean=df_fairs['BASKET_MINUS_SYNTHETIC'].mean(), lookback=400, ewma=5)

In [ ]:
def generate_signals(data, mean_price):
    """
    Generate trading signals based on the identified regimes and Bollinger Bands.

    Parameters:
    data (pd.DataFrame): The input DataFrame containing Bollinger Bands and regimes.
    mean_price (float): The mean price to compare against when entering riding regimes.

    Returns:
    pd.Series: A series containing the generated trading signals for each row.
    """
    price = data['BASKET_MINUS_SYNTHETIC']
    upper = data['UPPER_BAND']
    lower = data['LOWER_BAND']
    regime = data['REGIME']

    signals = pd.Series(index=data.index, dtype=object)

    for i in tqdm(range(1, len(data))):
        if regime[i] == 'OSCILLATING':
            if price[i] >= upper[i] and price[i-1] < upper[i-1]:
                signals[i] = 'SHORT'
            elif i < len(data) - 1 and price[i] >= upper[i] and price[i+1] < upper[i+1]:
                signals[i] = "SHORT"
            elif price[i] <= lower[i] and price[i-1] > lower[i-1]:
                signals[i] = 'LONG'
            elif i < len(data) - 1 and price[i] <= lower[i] and price[i+1] > lower[i+1]:
                signals[i] = 'LONG'
        elif regime[i] == 'NEUTRAL' and regime[i-1] != 'NEUTRAL':
            signals[i] = 'CLEAR'
        elif regime[i] == 'RIDING_UPPER' and regime[i-1] != 'RIDING_UPPER':
            if price[i] > mean_price:
                signals[i] = 'CLEAR'
            else:
                signals[i] = 'LONG'
        elif regime[i] == 'RIDING_LOWER' and regime[i-1] != 'RIDING_LOWER':
            if price[i] < mean_price:
                signals[i] = 'CLEAR'
            else:
                signals[i] = 'SHORT'

    return signals

df_fairs['SIGNAL'] = generate_signals(df_fairs, df_fairs['BASKET_MINUS_SYNTHETIC'].mean())

In [ ]:
fig = go.Figure()

# Add the BASKET_MINUS_SYNTHETIC line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['BASKET_MINUS_SYNTHETIC'],
                         mode='lines', name='BASKET_MINUS_SYNTHETIC'))

# Add the Upper Bollinger Band line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['UPPER_BAND'],
                         mode='lines', name='Upper Band', line=dict(color='red')))

# Add the Simple Moving Average (SMA) line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['SMA'],
                         mode='lines', name='SMA', line=dict(color='blue')))

# Add the Lower Bollinger Band line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['LOWER_BAND'],
                         mode='lines', name='Lower Band', line=dict(color='green')))

# Find regime switch points
regime_switch_points = df_fairs[df_fairs['REGIME'] != df_fairs['REGIME'].shift(1)]

# Add vertical lines for regime switches
for _, row in regime_switch_points.iterrows():
    fig.add_shape(type='line',
                  x0=row['timestamp'], y0=df_fairs['BASKET_MINUS_SYNTHETIC'].min(),
                  x1=row['timestamp'], y1=df_fairs['BASKET_MINUS_SYNTHETIC'].max(),
                  line=dict(color='black', width=1, dash='dash'))
    
    # Add hover text for regime switches
    fig.add_trace(go.Scatter(x=[row['timestamp']], y=[df_fairs.loc[row.name, 'BASKET_MINUS_SYNTHETIC']],
                             mode='markers', marker=dict(size=0), hoverinfo='text',
                             text=f"Regime: {row['REGIME']}<br>Last Sell Signal: {row['SIGNAL']}"))

# Update the layout
fig.update_layout(title='Bollinger Bands with Regime Switches',
                  xaxis_title='Timestamp',
                  yaxis_title='BASKET_MINUS_SYNTHETIC')

# Display the plot
fig.show()

## backtest

In [ ]:
df_fairs['SIGNAL'].value_counts()

In [ ]:
def update_position(signals):
    """
    Update the position based on the last non-nan signal.
    
    Parameters:
    signals (pd.Series): The series containing the trading signals.
    
    Returns:
    pd.Series: A series containing the updated positions.
    """
    last_signal = None
    positions = []
    
    for signal in signals:
        if pd.notna(signal):
            last_signal = signal
        
        if last_signal == 'LONG':
            position = 60
        elif last_signal == 'SHORT':
            position = -60
        else:
            position = 0
        
        positions.append(position)
    
    return pd.Series(positions, index=signals.index)

# Update the position based on the trading signals
df_fairs['POSITION'] = update_position(df_fairs['SIGNAL'])

In [ ]:
# Initialize the "CASH" column with zeros
df_fairs['CASH'] = 0

# Calculate the position differences
df_fairs['POSITION_DIFF'] = df_fairs['POSITION'].diff().fillna(0)

# Update the "CASH" column based on position changes
df_fairs['CASH'] = -df_fairs['POSITION_DIFF'] * df_fairs['BASKET_MINUS_SYNTHETIC']

# Calculate the cumulative cash
df_fairs['CUMULATIVE_CASH'] = df_fairs['CASH'].cumsum()

df_fairs['PNL'] = df_fairs['CUMULATIVE_CASH'] + df_fairs['POSITION'] * df_fairs["BASKET_MINUS_SYNTHETIC"]

In [ ]:
df_fairs[df_fairs['POSITION_DIFF'] != 0]

In [ ]:
df_fairs['POSITION'].value_counts()

In [ ]:
import plotly.graph_objects as go

# Create the plot
fig = go.Figure()

# Add the CUMULATIVE_CASH line
fig.add_trace(go.Scatter(x=df_fairs['timestamp'], y=df_fairs['PNL'],
                         mode='lines', name='Cum'))

# Update the layout
fig.update_layout(title='Cumulative Cash Over Time',
                  xaxis_title='Timestamp',
                  yaxis_title='Cumulative Cash')

# Display the plot
fig.show()

# gridsearch

In [ ]:
import itertools

def run_backtest(data, bb_window, bb_std, lookback):
    # Calculate Bollinger Bands
    data = bollinger_bands(data, window=bb_window, num_std=bb_std)

    # Identify the regime
    data['REGIME'] = identify_regime(data, mean=data['BASKET_MINUS_SYNTHETIC'].mean(), lookback=lookback)

    # Generate trading signals
    mean_price = data['BASKET_MINUS_SYNTHETIC'].mean()
    data['SIGNAL'] = generate_signals(data, mean_price)

    # Update the position based on the trading signals
    data['POSITION'] = update_position(data['SIGNAL'])

    # Initialize the "CASH" column with zeros
    data['CASH'] = 0

    # Calculate the position differences
    data['POSITION_DIFF'] = data['POSITION'].diff().fillna(0)

    # Update the "CASH" column based on position changes
    data['CASH'] = -data['POSITION_DIFF'] * data['BASKET_MINUS_SYNTHETIC']

    # Calculate the cumulative cash
    data['CUMULATIVE_CASH'] = data['CASH'].cumsum()

    # Calculate the PNL
    data['PNL'] = data['CUMULATIVE_CASH'] + data['POSITION'] * data["BASKET_MINUS_SYNTHETIC"]

    return data

# Define the arrays of candidate values for each parameter
bb_windows = [ 75, 100, 150, 200, 250, 300, 350, 400, 500]
bb_stds = [1.5,1.75, 2,2.25, 2.5]
lookbacks = [50,75,100, 125, 150, 175, 200]

# Perform the grid search
results = []
for bb_window, bb_std, lookback in itertools.product(bb_windows, bb_stds, lookbacks):
    # Run the backtest with the current parameter values
    data = run_backtest(df_fairs.copy(), bb_window, bb_std, lookback)

    # Get the final PNL
    final_pnl = data['PNL'].iloc[-1]

    # Store the results
    results.append({
        'bb_window': bb_window,
        'bb_std': bb_std,
        'lookback': lookback,
        'pnl': final_pnl
    })

    # Print the parameters and PNL for each iteration
    print(f"Bollinger Bands Window: {bb_window}, Bollinger Bands Std: {bb_std}, Lookback: {lookback}, PNL: {final_pnl}")

# Find the best parameters based on the highest PNL
best_result = max(results, key=lambda x: x['pnl'])
print("\nBest Parameters:")
print(f"Bollinger Bands Window: {best_result['bb_window']}")
print(f"Bollinger Bands Std: {best_result['bb_std']}")
print(f"Lookback: {best_result['lookback']}")
print(f"PNL: {best_result['pnl']}")